# 10_b - Match DBLP papers to OpenAlex (fallbacks)

**Purpose:** Re-run and extend the DBLP main-track paper universe produced in Notebook 09 to OpenAlex works. This notebook does **not** create an author-level table or label authors junior/senior.

**Inputs:** `output/dblp_papers_raw.csv` from Notebook 09.

**Outputs:**
- `output/dblp_openalex_matched.csv` — accepted paper-level matches
- `output/dblp_openalex_ambiguous.csv` — plausible but unaccepted candidates
- `output/dblp_openalex_unmatched.csv` — no credible candidate
- `output/dblp_openalex_match_report.csv` — coverage by conference-year
- `output/dblp_openalex_checkpoint.csv` — resumable processing state

Run locally or in Colab with internet access. Set `OPENALEX_API_KEY` as an environment variable if available; it is never written to disk.

In [12]:
import os, re, json, time, unicodedata
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd
import requests

INPUT_PATH = Path("output/dblp_papers_raw.csv")
OUTPUT_DIR = Path("output")
CHECKPOINT_PATH = OUTPUT_DIR / "dblp_openalex_checkpoint.csv"

MAILTO = ""  # Optional: your email address for OpenAlex polite-pool requests
API_KEY = "G0ErBTn3I9KHeI8oYZv0pd"

API_URL = "https://api.openalex.org/works"
PER_PAGE = 10
SLEEP_SECONDS = 0.12
YEAR_TOLERANCE = 1
AUTO_ACCEPT_SCORE = 0.94
AMBIGUOUS_SCORE = 0.82

OUTPUT_DIR.mkdir(exist_ok=True)
assert INPUT_PATH.exists(), f"Missing {INPUT_PATH}. Run Notebook 09 first."
assert API_KEY != "PASTE_YOUR_OPENALEX_API_KEY_HERE", "Paste your OpenAlex API key above."
print(f"Input: {INPUT_PATH}")
print("OpenAlex key detected:", bool(API_KEY))

Input: output\dblp_papers_raw.csv
OpenAlex key detected: True


## Load and audit input

The notebook expects at least `conference`, `year`, `title`, and `authors`. It creates a stable `dblp_row_id` so matching can be restarted safely.

In [13]:
papers = pd.read_csv(INPUT_PATH)
required = {"conference", "year", "title", "authors"}
missing = required - set(papers.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

papers = papers.copy()
papers["conference"] = papers["conference"].astype(str).str.upper().str.strip()
papers["year"] = pd.to_numeric(papers["year"], errors="coerce").astype("Int64")
papers["title"] = papers["title"].fillna("").astype(str).str.strip()
papers["authors"] = papers["authors"].fillna("").astype(str).str.strip()
papers = papers.loc[papers["title"].ne("") & papers["year"].notna()].reset_index(drop=True)
papers["dblp_row_id"] = [f"DBLP_{i:07d}" for i in range(1, len(papers) + 1)]

print(f"Papers available: {len(papers):,}")
print(f"Duplicate conference-year-title rows: {papers.duplicated(['conference','year','title']).sum():,}")
papers.groupby("conference").size().sort_values(ascending=False).head(10)

Papers available: 83,319
Duplicate conference-year-title rows: 36


conference
CVPR       8458
AAAI       7004
NIPS       6502
IJCAI      5612
CHI        5522
INFOCOM    5499
CIKM       4477
ICML       4106
SIGIR      3501
KDD        3026
dtype: int64

## Matching functions

Candidates come from OpenAlex title search. A candidate is scored from normalized-title similarity, year agreement, and surname overlap. Candidate metadata and scoring fields are retained for auditability.

In [15]:
class OpenAlexQuotaError(RuntimeError):
    """Raised when OpenAlex returns HTTP 429."""


def normalize_text(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def title_similarity(left, right):
    return SequenceMatcher(
        None,
        normalize_text(left),
        normalize_text(right)
    ).ratio()


def surname_set(authors):
    names = re.split(r"\s*(?:,|;| and )\s*", str(authors))
    result = set()

    for name in names:
        tokens = normalize_text(name).split()
        if tokens:
            result.add(tokens[-1])

    return result


def work_surnames(work):
    names = []

    for authorship in work.get("authorships", []) or []:
        name = ((authorship.get("author") or {}).get("display_name"))
        if name:
            names.append(name)

    return surname_set("; ".join(names))


def year_score(dblp_year, work_year):
    if work_year is None:
        return 0.0

    gap = abs(int(dblp_year) - int(work_year))

    if gap == 0:
        return 1.0
    if gap <= YEAR_TOLERANCE:
        return 0.75

    return 0.0


def candidate_score(row, work):
    title_score = title_similarity(row.title, work.get("title") or "")
    y_score = year_score(row.year, work.get("publication_year"))

    db_surnames = surname_set(row.authors)
    oa_surnames = work_surnames(work)

    overlap = (
        len(db_surnames & oa_surnames) / len(db_surnames)
        if db_surnames
        else 0.0
    )

    total = (
        0.78 * title_score
        + 0.14 * y_score
        + 0.08 * overlap
    )

    return {
        "match_score": round(total, 4),
        "title_score": round(title_score, 4),
        "year_score": round(y_score, 4),
        "author_overlap": round(overlap, 4),
    }


def clean_search_query(text, max_length=300):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("*", " ")
    text = text.replace("?", " ")
    text = text.replace(":", " ")
    text = text.replace(";", " ")
    text = text.replace('"', " ")
    text = text.replace("'", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text[:max_length]


def fetch_candidates(query, retries=3):
    safe_query = clean_search_query(query)

    params = {
        "search": safe_query,
        "per-page": PER_PAGE,
        "select": (
            "id,doi,title,display_name,publication_date,publication_year,"
            "cited_by_count,authorships,primary_location,type"
        ),
    }

    if MAILTO:
        params["mailto"] = MAILTO

    if API_KEY:
        params["api_key"] = API_KEY

    last_error = None

    for attempt in range(retries):
        try:
            response = requests.get(API_URL, params=params, timeout=45)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After", "unknown")

                raise OpenAlexQuotaError(
                    "OpenAlex returned HTTP 429. Rate limit or daily credits "
                    f"may be exhausted. Retry-After: {retry_after}. "
                    "The checkpoint will be saved before the notebook stops."
                )

            response.raise_for_status()
            time.sleep(SLEEP_SECONDS)

            return response.json().get("results", [])

        except OpenAlexQuotaError:
            raise

        except requests.RequestException as exc:
            last_error = exc

            if attempt < retries - 1:
                wait_seconds = 2 ** attempt
                print(
                    f"Request failed ({type(exc).__name__}); "
                    f"retrying in {wait_seconds}s..."
                )
                time.sleep(wait_seconds)

    raise last_error


def fallback_query(row):
    clean_title = normalize_text(row.title)
    surnames = sorted(surname_set(row.authors))[:3]

    return " ".join([clean_title, *surnames])


def get_scored_candidates(row):
    """Search title first; add author surnames only for weak results."""
    title_candidates = fetch_candidates(row.title)

    scored = [
        (work, candidate_score(row, work))
        for work in title_candidates
    ]
    scored.sort(key=lambda item: item[1]["match_score"], reverse=True)

    best_score = scored[0][1]["match_score"] if scored else 0.0

    if best_score < AMBIGUOUS_SCORE:
        try:
            fallback_candidates = fetch_candidates(fallback_query(row))

            by_id = {
                work.get("id"): work
                for work, _ in scored
                if work.get("id")
            }

            by_id.update({
                work.get("id"): work
                for work in fallback_candidates
                if work.get("id")
            })

            scored = [
                (work, candidate_score(row, work))
                for work in by_id.values()
            ]
            scored.sort(
                key=lambda item: item[1]["match_score"],
                reverse=True
            )

        except OpenAlexQuotaError:
            raise

        except requests.RequestException:
            pass

    return scored


def flatten_result(row, work, score, status, rank):
    primary_source = (
        ((work.get("primary_location") or {}).get("source") or {})
        .get("display_name")
    )

    row_dict = row._asdict() if hasattr(row, "_asdict") else row.to_dict()

    return {
        **row_dict,
        **score,
        "match_status": status,
        "candidate_rank": rank,
        "openalex_id": work.get("id"),
        "openalex_doi": work.get("doi"),
        "openalex_title": work.get("title") or work.get("display_name"),
        "openalex_publication_date": work.get("publication_date"),
        "openalex_publication_year": work.get("publication_year"),
        "openalex_cited_by_count": work.get("cited_by_count"),
        "openalex_primary_source": primary_source,
        "openalex_type": work.get("type"),
        "openalex_authorships": json.dumps(
            work.get("authorships", []),
            ensure_ascii=False
        ),
    }

## Resume-aware matching run

Begin with `MAX_ROWS = 100` to validate matches, inspect the ambiguous output, then set it to `None` for the full run. Previously checkpointed rows are skipped.

In [16]:
MAX_ROWS = None  # Keep at 100 for validation; set to None for the full corpus.

existing = pd.read_csv(CHECKPOINT_PATH) if CHECKPOINT_PATH.exists() else pd.DataFrame()
# Keep accepted matches; retry prior unresolved/API-error rows with the new fallback logic.
accepted_ids = set(existing.loc[existing["match_status"].eq("matched"), "dblp_row_id"].astype(str)) if not existing.empty else set()
todo = papers.loc[~papers["dblp_row_id"].isin(accepted_ids)].copy()
if MAX_ROWS is not None:
    todo = todo.head(MAX_ROWS)

new_rows = []
for position, row in enumerate(todo.itertuples(index=False), start=1):
    try:
        scored = get_scored_candidates(row)
        if not scored:
            new_rows.append({**row._asdict(), "match_status":"unmatched", "match_score":0.0,
                             "title_score":0.0, "year_score":0.0, "author_overlap":0.0,
                             "candidate_rank":None, "openalex_id":None, "openalex_doi":None,
                             "openalex_title":None, "openalex_publication_date":None,
                             "openalex_publication_year":None, "openalex_cited_by_count":None,
                             "openalex_primary_source":None, "openalex_type":None,
                             "openalex_authorships":None, "error":None})
        else:
            best_work, best_score = scored[0]
            margin = best_score["match_score"] - (scored[1][1]["match_score"] if len(scored) > 1 else 0)
            status = "matched" if best_score["match_score"] >= AUTO_ACCEPT_SCORE and margin >= 0.03 else "ambiguous" if best_score["match_score"] >= AMBIGUOUS_SCORE else "unmatched"
            new_rows.append(flatten_result(row, best_work, best_score, status, 1))
        if position % 25 == 0:
            print(f"Processed {position:,}/{len(todo):,}")
    except requests.RequestException as exc:
        new_rows.append({**row._asdict(), "match_status":"api_error", "match_score":None,
                         "title_score":None, "year_score":None, "author_overlap":None,
                         "candidate_rank":None, "openalex_id":None, "openalex_doi":None,
                         "openalex_title":None, "openalex_publication_date":None,
                         "openalex_publication_year":None, "openalex_cited_by_count":None,
                         "openalex_primary_source":None, "openalex_type":None,
                         "openalex_authorships":None, "error":str(exc)})

# New attempts replace older unresolved rows rather than creating duplicate paper records.
new_results = pd.DataFrame(new_rows)
if existing.empty:
    updated = new_results
else:
    updated = pd.concat([
        existing.loc[~existing["dblp_row_id"].isin(new_results["dblp_row_id"])],
        new_results
    ], ignore_index=True)
updated.to_csv(CHECKPOINT_PATH, index=False)
print(f"Checkpoint rows: {len(updated):,}; retried/newly processed: {len(new_rows):,}")

OpenAlexQuotaError: OpenAlex returned HTTP 429. Rate limit or daily credits may be exhausted. Retry-After: 17385. The checkpoint will be saved before the notebook stops.

## Write final outputs and coverage report

Only `matched` rows are treated as accepted matches. Ambiguous rows remain separate for review; they are not silently included in later analysis.

In [9]:
results = pd.read_csv(CHECKPOINT_PATH)
matched = results.loc[results.match_status.eq("matched")].copy()
ambiguous = results.loc[results.match_status.eq("ambiguous")].copy()
unmatched = results.loc[results.match_status.isin(["unmatched", "api_error"])].copy()

matched.to_csv(OUTPUT_DIR / "dblp_openalex_matched.csv", index=False)
ambiguous.to_csv(OUTPUT_DIR / "dblp_openalex_ambiguous.csv", index=False)
unmatched.to_csv(OUTPUT_DIR / "dblp_openalex_unmatched.csv", index=False)

report = (results.assign(processed=1, matched_flag=results.match_status.eq("matched").astype(int))
          .groupby(["conference", "year"], dropna=False)
          .agg(papers=("processed", "sum"), matched=("matched_flag", "sum"),
               ambiguous=("match_status", lambda s: (s == "ambiguous").sum()),
               unmatched=("match_status", lambda s: s.isin(["unmatched", "api_error"]).sum()),
               mean_match_score=("match_score", "mean"))
          .reset_index())
report["match_rate"] = report["matched"] / report["papers"]
report.to_csv(OUTPUT_DIR / "dblp_openalex_match_report.csv", index=False)

print(results.match_status.value_counts(dropna=False))
print(f"Accepted match rate among processed rows: {len(matched)/len(results):.1%}" if len(results) else "No rows processed.")
report.sort_values(["match_rate", "papers"]).head(15)

match_status
matched      244
unmatched     25
ambiguous      6
Name: count, dtype: int64
Accepted match rate among processed rows: 88.7%


,conference,year,papers,matched,ambiguous,unmatched,mean_match_score,match_rate
0,AAAI,2000,233,205,4,24,0.938165,0.879828
1,AAAI,2002,42,39,2,1,0.985405,0.928571


## Manual review checklist

Before proceeding to author extraction in Notebook 11:
- Inspect a random sample of accepted matches, especially short or generic titles.
- Inspect all ambiguous rows with high scores.
- Re-run this notebook with `MAX_ROWS = None` only after the scoring thresholds look reliable.
- Keep the checkpoint and all three result files as the paper-level matching audit trail.

In [6]:
results = pd.read_csv(CHECKPOINT_PATH)

review = (
    results.loc[
        results["match_status"].isin(["unmatched", "ambiguous", "api_error"]),
        [
            "dblp_row_id",
            "title",
            "authors",
            "match_status",
            "match_score",
            "title_score",
            "year_score",
            "author_overlap",
            "openalex_title",
            "openalex_publication_year",
            "error",
        ],
    ]
    .sort_values(["match_status", "match_score"], na_position="last")
)

display(review)

,dblp_row_id,title,authors,match_status,match_score,title_score,year_score,author_overlap,openalex_title,openalex_publication_year,error
97,DBLP_0000049,Assessing Relevance with Extensionally Defined...,Bruce M. McLaren; Kevin D. Ashley,ambiguous,0.8600,1.0000,0.0,1.0000,Assessing Relevance With Extensionally Defined...,2018.0,NaN
98,DBLP_0000054,Compilability of Abduction.,Paolo Liberatore; Marco Schaerf,ambiguous,0.8600,1.0000,0.0,1.0000,Compilability of Abduction,2002.0,NaN
95,DBLP_0000018,Self-Organization of Innate Face Preferences: ...,James A. Bednar; Risto Miikkulainen,api_error,NaN,NaN,NaN,NaN,NaN,NaN,400 Client Error: Bad Request for url: https:/...
135,DBLP_0000136,Localizing A*.,Stefan Edelkamp; Stefan Schrödl,api_error,NaN,NaN,NaN,NaN,NaN,NaN,400 Client Error: Bad Request for url: https:/...
141,DBLP_0000142,A*with Partial Expansion for Large Branching F...,Takayuki Yoshizumi; Teruhisa Miura; Toru Ishida,api_error,NaN,NaN,NaN,NaN,NaN,NaN,400 Client Error: Bad Request for url: https:/...
99,DBLP_0000065,Total Knowledge.,Ian Pratt-Hartmann,unmatched,0.3069,0.3934,0.0,0.0000,Maintaining knowledge about temporal intervals,1983.0,NaN
188,DBLP_0000189,Model-Based-Diagnosis for Fault Management in ...,Aomar Osmani,unmatched,0.3317,0.4253,0.0,0.0000,Research Status and Challenges of Data‐Driven ...,2021.0,NaN
185,DBLP_0000186,A Semi-Complete Disambiguation Algorithm for O...,Rada Mihalcea,unmatched,0.3578,0.4587,0.0,0.0000,Learning morphological disambiguation rules fo...,2006.0,NaN
137,DBLP_0000138,Change Detection in Heuristic Search.,Eyke Hüllermeier,unmatched,0.3756,0.4815,0.0,0.0000,An application of heuristic search methods to ...,1976.0,NaN
182,DBLP_0000183,Tracing Dependencies of Strategy Selections in...,Dung N. Lam; K. Suzanne Barber,unmatched,0.3934,0.5043,0.0,0.0000,Heuristics as adaptive decision strategies in ...,2014.0,NaN
